#### Import libraries

In [ ]:
from agents import Agent, Runner, trace
import os, asyncio
from dotenv import load_dotenv

load_dotenv(override=True)

#### Define Agents

In [ ]:
# Agent 1: Summarizer
summarizer_agent = Agent(
    name="BioSummarizer",
    instructions="Summarize the given biomedical abstract in 50 words or less.",
    model="gpt-4.1-nano"
)

# Agent 2: Critique/Reviewer
reviewer_agent = Agent(
    name="SummaryReviewer",
    instructions=(
        "Review the provided summary of a biomedical abstract. "
        "If the summary is accurate, concise, covers the main biomedical points, and is clearly about biomedicine, reply 'ACCEPTED'. "
        "If the summary is not about a biomedical topic, or is missing key biomedical points, reply 'REJECTED' stating briefly why in 15 words or less."
    ),
    model="gpt-4.1-nano"
)


#### Define Reflective workflow

In [ ]:
async def reflection_pipeline(abstract, max_attempts=3):
    for attempt in range(max_attempts):
        await asyncio.sleep(30)
        # Step 1: Summarize
        summary_result = await Runner.run(summarizer_agent, abstract)
        summary = summary_result.final_output
        print(f"Attempt {attempt+1} Summary: {summary}")

        # Step 2: Critique
        await asyncio.sleep(30)
        review_prompt = f"Abstract: {abstract}\nSummary: {summary}"
        review_result = await Runner.run(reviewer_agent, review_prompt)
        review = review_result.final_output
        print(f"Review: {review}")

        if "ACCEPTED" in review:
            print("Summary accepted.")
            return summary
        else:
            print("Summary rejected. Revising...")

    print("Failed to generate an acceptable summary after several attempts.")
    return None


#### Pass Example

In [ ]:
with trace("reflection_pipeline_pass"):
    abstract = (
        "BRCA1 and TP53 are important genes in cancer biology. "
        "EGFR is also frequently mutated. These genes play a role in cell cycle regulation and tumor suppression."
    )
    final_summary = await reflection_pipeline(abstract)
    print("Final Summary:", final_summary)

#### Fail Example

In [ ]:
# Example usage
with trace("reflection_pipeline_fail"):
   abstract = (
        "The S&P 500 index rose by 2% last quarter, driven by gains in technology stocks. "
        "Investors remain optimistic about the Federal Reserve's interest rate policy. "
        "However, inflation concerns persist and may impact future market performance."
    )
   final_summary = await reflection_pipeline(abstract)
   print("Final Summary:", final_summary)